# Prednost domačega terena in boj za naslov v Premier League (1992–2026)

Projektna naloga pri predmetu **Uvod v programiranje** (FMF, Univerza v Ljubljani).

V zvezku analiziram podatke o vseh tekmah in končnih lestvicah angleške
Premier League od sezone 1992/93 do sezone 2025/26 (34 sezon). Podatke sem
zajel s spletne strani [FBref.com](https://fbref.com/en/comps/9/Premier-League-Stats)
(postopek je opisan v `README.md`).

Odgovoriti želim na dve skupini vprašanj:

1. **Ali domači teren vpliva na uspeh?** Kako pogosto zmagajo domači, ali dajo
   več golov, kaj se je zgodilo, ko so bile tribune med epidemijo covida-19
   prazne, in ali več gledalcev pomeni več zmag.
2. **Kdo osvaja ligo in koliko točk je za to potrebnih?** Kateri klubi so
   najpogosteje na vrhu in na dnu lestvice, koliko točk potrebuje prvak,
   koliko jih je dovolj za obstanek in kako velika je razlika med vrhom in dnom.

Daljša koda (nalaganje, izračuni in grafi) je v datoteki `analiza_funkcije.py`,
da je zvezek pregleden.

In [17]:
import matplotlib.pyplot as plt
import pandas as pd

from analiza_funkcije import (
    delez_izidov, delez_izidov_po_sezonah, graf_golov_po_sezonah,
    graf_izidov_po_sezonah, manjkajoce_vrednosti, nalozi_lestvice, nalozi_tekme,
    povprecni_goli_po_sezonah, pregled_po_sezonah, nastavi_slog,
)

# Da pandas izpiše vse stolpce tabele, ne le prvih in zadnjih.
pd.set_option("display.max_columns", None)

# Enoten slog vseh grafov (barve, mreža, pisava).
nastavi_slog()

ImportError: cannot import name 'delez_izidov' from 'analiza_funkcije' (c:\Users\aljo\Desktop\uvp_project\projekt-uvp\analiza_funkcije.py)

## 1. Uvoz podatkov

Podatki so v dveh CSV datotekah v mapi `podatki/`, ki ju ustvari `py main.py`:

- `tekme.csv` – ena vrstica za vsako odigrano tekmo,
- `lestvice.csv` – ena vrstica za vsak klub na končni lestvici vsake sezone.

In [ ]:
tekme = nalozi_tekme()
lestvice = nalozi_lestvice()

print(f"Tekme:    {tekme.shape[0]} vrstic, {tekme.shape[1]} stolpcev")
print(f"Lestvice: {lestvice.shape[0]} vrstic, {lestvice.shape[1]} stolpcev")

### Tekme

| stolpec | pomen |
|---|---|
| `sezona` | sezona, npr. `2023-2024` |
| `krog` | številka kroga |
| `dan`, `datum`, `ura` | dan v tednu, datum in ura začetka (angleški čas) |
| `domaci`, `gostje` | domača in gostujoča ekipa |
| `goli_domaci`, `goli_gostje` | doseženi goli |
| `izid` | `D` zmaga domačih, `X` remi, `G` zmaga gostov |
| `gledalci` | število gledalcev |
| `stadion`, `sodnik` | stadion in glavni sodnik |
| `opomba` | opomba FBrefa k tekmi |

In [ ]:
tekme.head()

### Lestvice

| stolpec | pomen |
|---|---|
| `sezona`, `mesto`, `klub` | sezona, končno mesto in klub |
| `tekme`, `zmage`, `remiji`, `porazi` | odigrane tekme in njihovi izidi |
| `dani_goli`, `prejeti_goli`, `gol_razlika` | goli |
| `tocke`, `tocke_na_tekmo` | točke na končni lestvici (že z morebitnimi odbitki) |
| `povprecje_gledalcev` | povprečno število gledalcev na domačih tekmah |
| `najboljsi_strelec`, `goli_strelca` | najboljši strelec kluba in njegovi goli |
| `opomba` | npr. uvrstitev v evropska tekmovanja, izpad ali odbitek točk |

In [ ]:
lestvice.head()

## 2. Pregled po sezonah

Vsak klub z vsakim drugim igra enkrat doma in enkrat v gosteh, zato ima
sezona z $n$ klubi $n(n-1)$ tekem. V prvih treh sezonah je bilo v ligi 22 klubov
(462 tekem), od sezone 1995/96 naprej pa 20 klubov (380 tekem).

In [ ]:
pregled = pregled_po_sezonah(lestvice, tekme)
pregled

In [ ]:
print(f"Število sezon: {tekme['sezona'].nunique()}")
print(f"Število tekem: {len(tekme)}")
print(f"Število golov: {pregled['goli'].sum()} ({pregled['goli'].sum() / len(tekme):.2f} na tekmo)")
print(f"Različnih klubov v ligi: {lestvice['klub'].nunique()}")

# Klubi, ki so v ligi igrali vseh 34 sezon.
sezon_v_ligi = lestvice.groupby("klub").size()
print("V ligi vse sezone:", ", ".join(sezon_v_ligi[sezon_v_ligi == tekme["sezona"].nunique()].index))

V ligi je skozi 34 sezon igral 51 različnih klubov, le šest jih ni nikoli izpadlo.
Število golov na tekmo je večino časa med 2,5 in 2,8 (najmanj 2,45 v sezoni
2006/07). Izstopa sezona 2023/24 z rekordnimi 3,28 gola na tekmo.

## 3. Manjkajoči podatki

Preden začnem z analizo, preverim, kje podatki manjkajo. Vse prazne vrednosti
manjkajo že na FBrefu in niso posledica napake pri zajemu (to sem preveril
tako, da sem točke, gole in število tekem, izračunane iz posameznih tekem,
primerjal s končnimi lestvicami).

In [ ]:
manjkajoce_vrednosti(tekme)

In [ ]:
manjkajoce_vrednosti(lestvice)

In [ ]:
# Kje manjkajo podatki o gledalcih (le sezone, kjer kaj manjka).
pregled.loc[pregled["brez_gledalcev"] > 0, ["tekme", "brez_gledalcev"]]

Razlage:

- **`ura`** – FBref ure začetka nima za tekme do vključno sezone 2013/14.
  Za analizo je ne potrebujem.
- **`gledalci`** – podatki manjkajo iz dveh različnih razlogov:
  - **covid-19:** v sezoni 2019/20 so vse tekme od nadaljevanja lige
    (17. 6. 2020) igrali brez gledalcev (92 tekem), v sezoni 2020/21 pa skoraj
    vse (348 tekem; na preostalih 32 je bilo dovoljenih 2.000 do 10.000 gledalcev),
  - **manjkajoči podatki:** FBref za sezono 2007/08 skoraj nima podatkov o
    gledalcih, za 2008/09 manjkajo za 141 tekem, drugje le za posamezne tekme.
    Tribune takrat niso bile prazne!
- **`povprecje_gledalcev`** – manjka pri 14 klubih iz sezone 2007/08 (isti razlog).
- **`opomba`** pri tekmah je prazna pri vseh tekmah, pri lestvicah pa jo imajo
  le klubi z evropsko uvrstitvijo, izpadom ali odbitkom točk.

**Posledice za analizo:** tekme brez gledalcev zaradi covida določim po praznem
podatku *samo v sezonah 2019/20 in 2020/21*. Pri analizi vpliva števila
gledalcev tekme brez podatka izpustim (ne štejem jih kot 0 gledalcev).

## 4. Prvi vpogled: izidi vseh tekem

Za uvod pogledam, kako so se končale vse tekme skupaj.

In [ ]:
imena_izidov = {"D": "zmaga domačih", "X": "remi", "G": "zmaga gostov"}
delez_izidov(tekme).rename(index=imena_izidov).round(1)

Domači zmagajo na skoraj polovici tekem (45,7 %), gostje pa le na dobri
četrtini (28,7 %). To nakazuje izrazito prednost domačega terena, ki jo podrobneje
raziščem v naslednjem razdelku.

## 5. Sklop 1: Ali domači teren vpliva na uspeh?

### 5.1 Zmage, remiji in porazi doma in v gosteh

Vsaka tekma ima domačo in gostujočo ekipo, zato zmaga domačih pomeni poraz
gostov in obratno. Za vsako sezono izračunam delež zmag domačih, remijev in
zmag gostov.

In [ ]:
delezi = delez_izidov_po_sezonah(tekme)
graf_izidov_po_sezonah(delezi)
plt.show()

In [ ]:
# Povprečje prvih in zadnjih desetih sezon (v odstotkih).
primerjava = pd.DataFrame({
    "prvih 10 sezon": delezi.head(10).mean(),
    "zadnjih 10 sezon": delezi.tail(10).mean(),
    "vse sezone": delez_izidov(tekme),
}).rename(index={"D": "zmaga domačih", "X": "remi", "G": "zmaga gostov"})
primerjava.round(1)

In [ ]:
# Koliko točk v povprečju osvoji ekipa na tekmo doma in v gosteh
# (zmaga 3 točke, remi 1 točka).
tocke_doma = (3 * (tekme["izid"] == "D") + (tekme["izid"] == "X")).mean()
tocke_v_gosteh = (3 * (tekme["izid"] == "G") + (tekme["izid"] == "X")).mean()
print(f"Točke na tekmo doma:     {tocke_doma:.2f}")
print(f"Točke na tekmo v gosteh: {tocke_v_gosteh:.2f}")

Domači so v **vsaki sezoni razen 2020/21** zmagali pogosteje kot gostje,
običajno na 41–51 % tekem, gostje pa na 24–35 %. Ekipa doma osvoji v povprečju
1,63 točke na tekmo, v gosteh pa le 1,12. Skozi čas se je prednost nekoliko
zmanjšala: delež zmag domačih je skoraj enak (45,8 % v prvih in 44,6 % v zadnjih
desetih sezonah), vendar je manj remijev (27,8 % → 23,2 %) in več zmag gostov
(26,5 % → 32,1 %). Izstopa sezona 2020/21, ko so bile tribune prazne: takrat so
gostje zmagali pogosteje kot domači (40,3 % proti 37,9 %).

### 5.2 Vpliv domačega terena na dane in prejete gole

Za vsako sezono izračunam, koliko golov so v povprečju na tekmo dali domači
in koliko gostje. Goli, ki jih dajo domači, so goli, ki jih prejmejo gostje,
zato graf hkrati kaže dane in prejete gole obeh strani.

In [ ]:
goli = povprecni_goli_po_sezonah(tekme)
graf_golov_po_sezonah(goli)
plt.show()

In [ ]:
print(f"Povprečno na tekmo: domači {tekme['goli_domaci'].mean():.2f}, "
      f"gostje {tekme['goli_gostje'].mean():.2f} gola")
print(f"Domači dajo {tekme['goli_domaci'].mean() / tekme['goli_gostje'].mean() - 1:.0%} več golov kot gostje.")

# Sezone z največjo in najmanjšo razliko med goli domačih in gostov.
goli["razlika"].round(2).sort_values().iloc[[0, 1, -1]]

Domači dajo v povprečju 1,53 gola na tekmo, gostje pa 1,16, torej približno
**32 % več**. Črta domačih je v vseh sezonah nad črto gostov, vendar se razlika
zmanjšuje: v prvih desetih sezonah je bila v povprečju 0,42 gola na tekmo, v
zadnjih desetih le še 0,28. Gostje so v zadnjem desetletju postali nevarnejši
(1,10 → 1,28 gola na tekmo), domači pa so ostali na skoraj enaki ravni.
V sezoni 2020/21 brez gledalcev je razlika skoraj izginila (0,01 gola), kar
nakazuje, da del prednosti domačega terena prinašajo prav gledalci. To podrobneje
preverim v naslednjem razdelku.